In [ ]:
####################################
# ENVIRONMENT SETUP

In [ ]:
# LIBRARIES

# system
import glob
import math
import os
import pickle
import sys

# datetime
from datetime import datetime, timedelta

import cartopy.crs as ccrs
import cartopy.feature as cfeature

# DownloadERA5Data functions
import cdsapi

# plotting
import matplotlib

# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt

# math and array operations
import numpy as np
import pandas as pd

# data classes
import xarray as xr
from matplotlib.colors import TwoSlopeNorm

# loading bar
from tqdm import tqdm

In [ ]:
# Importing DirectoryManager Class
sys.path.append(
    os.path.join(
        "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/",
        "DataAnalysis",
    )
)
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "ERA5_Data")
dataType = "ERA5Comparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(
    codeType, dataType
)

In [ ]:
# Importing ModelData Class
sys.path.append(
    os.path.join(DirectoryManager.mainCodeDirectory, "DataAnalysis", "MPAS_Model_Data")
)
from CLASSES_ModelData import DataOperator_Class, StructuredModelData_Class

In [ ]:
spinup_hours = "0"
RunType = ("TRACER","WET","NSSL",spinup_hours)

# spinup_hours = "-6"
# RunType = ("TRACER","DIURNAL","NSSL",spinup_hours)

ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
# Importing ERA5 Data Loading Classes
sys.path.append(
    os.path.join(DirectoryManager.mainCodeDirectory, "DataAnalysis", "ERA5_Data")
)
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class

In [ ]:
####################################
# DOWNLOADING SURFACE DATA

In [ ]:
variables = ERA5DataLoading_Class.GetVariableNames_Surface()
date = ERA5DataLoading_Class.GetERA5Date(ModelData.SimulationTime)
area = ERA5DataLoading_Class.GetERA5Area(ModelData.latitude, ModelData.longitude)
ERA5FilePath = ERA5DataLoading_Class.GetERA5FilePath(DirectoryManager, ModelData)

In [ ]:
ERA5DataLoading_Class.DownloadERA5_Surface(variables, date, area, ERA5FilePath)

In [ ]:
####################################
# MAKING ACCUMULATED RAIN DATA

In [ ]:
def LoadERA5_AccumulatedPrecip(DirectoryManager, ModelData):
    # Load ERA5 full data once
    ERA5_tp = ERA5DataLoading_Class.LoadERA5Data(
        DirectoryManager,
        ModelData,
        variableName="total_precipitation",
        dataType="Surface",
    ).load()

    # Compute cumulative precipitation (matching MPAS rainnc)
    ERA5_cumulative = ERA5_tp.cumsum(dim="valid_time")
    ERA5_cumulative*=1000
    ERA5_cumulative.attrs["long_name"] = "Accumulated total precipitation"

    ERA5_cumulative = ERA5_cumulative.assign_coords(
        longitude=((ERA5_cumulative.longitude + 360) % 360)
    )

    return ERA5_cumulative


def SaveERA5_AccumulatedPrecip(DirectoryManager, ModelData, ERA5_cumulative):
    ERA5FilePath = ERA5DataLoading_Class.GetERA5FilePath(DirectoryManager, ModelData)
    outputFilePath = os.path.join(
        ERA5FilePath, "ERA5Surface_accumulated_precipitation.nc"
    )
    ds = ERA5_cumulative.to_dataset(name="tp")
    ds.to_netcdf(outputFilePath)
    print(f"Saved to {outputFilePath}")

In [ ]:
# DOWNLOADING SURFACE DATA

ERA5_cumulative = LoadERA5_AccumulatedPrecip(DirectoryManager, ModelData)
SaveERA5_AccumulatedPrecip(DirectoryManager, ModelData, ERA5_cumulative)